# Lesson 10 - AI-agenter i produktion

I denna lektion kommer du att lära dig **produktionsmönster** för AI-agenter med Microsoft Agent Framework (Python). Vi täcker:

- **Observerbarhet** — lägga till tidtagning och loggning av agentinteraktioner
- **Utvärdering** — använda en utvärderingsagent för att bedöma svarskvalitet
- **Kostnadshantering** — strategier för tokenoptimering och modellval

Scenariot är en **resebyrå** som hjälper användare att planera resor, med övervakning och utvärdering ovanpå.


## Installation


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Produktionsöverväganden

Att flytta AI-agenter från prototyper till produktion kräver noggrann uppmärksamhet på tre pelare:

1. **Observerbarhet** — Du behöver insyn i vad agenten gör, hur lång tid det tar och vilka verktyg den anropar. Utan spårning och loggning är det nästan omöjligt att felsöka problem i produktion.

2. **Utvärdering** — Automatiserade kvalitetskontroller säkerställer att agentens svar förblir korrekta, fullständiga och hjälpsamma över tid. En utvärderingsagent kan poängsätta svar mot definierade kriterier.

3. **Kostnadshantering** — Användningen av tokens påverkar direkt kostnaden. Strategier som optimering av promptar, val av modell och cachning hjälper till att hålla kostnaderna under kontroll utan att offra kvalitet.


## Bygga en Observerbar Agent

Vi definierar resverktyg och omsluter agentanropet med tidsmätning så att vi kan övervaka fördröjning. I produktion skulle du integrera med OpenTelemetry eller en liknande spårningsbackend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Utvärderingsmönster

Ett vanligt produktionsmönster är att använda en andra agent som en **utvärderare**. Utvärderaren betygsätter den primära agentens svar utifrån fördefinierade kriterier såsom fullständighet, noggrannhet och hjälpsamhet.

Detta möjliggör:
- Automatiska kvalitetskontroller innan svar når användare
- Regressiondetektion när prompts eller modeller ändras
- Kontinuerlig övervakning av agentens prestanda över tid


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kostnadshanteringsstrategier

Att kontrollera kostnader är avgörande för produktions-AI-agenter. Här är viktiga strategier:

| Strategy | Description |
|---|---|
| **Prompt optimization** | Håll systeminstruktioner kortfattade. Ta bort överflödig kontext för att minska inmatningstokener. |
| **Model selection** | Använd mindre, billigare modeller (t.ex. GPT-4o-mini) för enkla uppgifter som klassificering eller extrahering, och reservera större modeller för komplexa resonemang. |
| **Caching** | Cachar verktygsresultat och frekventa frågor för att undvika onödiga API-anrop. |
| **Token budgets** | Sätt `max_tokens`-gränser för att förhindra oväntat långa svar. |
| **Batching** | Gruppera flera användarfrågor i ett enda API-anrop när det är möjligt. |

I praktiken fungerar en flernivåstrategi bra: dirigera enkla förfrågningar till en snabb, billig modell och eskalera endast komplexa frågor till en mer kapabel.


## Sammanfattning

I denna lektion lärde du dig hur du:

1. **Lägger till observabilitet** i agentinteraktioner med tidtagning och loggning, vilket lägger grunden för spårning och övervakning.
2. **Utvärderar agenters svar** automatiskt med en utvärderingsagent som bedömer fullständighet, noggrannhet och hjälpsamhet.
3. **Hantera kostnader** genom optimering av promptar, val av modell, caching och token-budgetar.

Dessa produktionsmönster hjälper till att säkerställa att dina AI-agenter är pålitliga, mätbara och kostnadseffektiva i stor skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
